# Loading dataframe and defining features and target

In [ ]:
from __future__ import print_function, division   # Ensures Python3 printing & division standard
import pandas as pd 
from pandas import Series, DataFrame 
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split 
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix
import lightgbm as lgb
from lightgbm import early_stopping
import time
import warnings

warnings.filterwarnings("ignore")

SavePlots = False

## Defining candidates for final feature set

In [23]:
data = pd.DataFrame(np.genfromtxt('data_mice_encoded.csv', delimiter=',', names=True))
data = data[data['CLAIM_COUNT'] == 1]

limit = 200000

variables = data.columns
target = 'CLAIM_SIZE_INDEX'

top10_feat = ['CONSTRUCTION_YEAR', 
              'HOUDEN10KM', 
              'DEDUCTIBLE', 
              'RESIDENTIAL_AREA', 
              'ROOF_TYPE_CAT_Thatched_Roof', 
              'BUILDINGS', 
              'WATER_SUPPLY_TYPE_CAT_Private_Water_Supply', 
              'BASEMENT_AREA', 
              'OUTER_WALLS_CAT_Brick_Walls', 
              'WATER_SUPPLY_TYPE_CAT_Water_Extraction_Plant']
top20_feat = top10_feat + [
    'OUTER_WALLS_CAT_Timber_Framing',
    'HEATING_TYPE_CAT_Central_Heating_Own_System',
    'OUTER_WALLS_CAT_Wood_Cladding', #highly corr w OUTER_WALLS_CAT_Brick_Walls
    'ROOF_TYPE_CAT_Fiber_Cement_Asbestos',
    'HEATING_TYPE_CAT_District_Heating', #highly corr w HEATING_TYPE_CAT_Central_Heating_Own_System
    'OUTER_WALLS_CAT_Lightweight_Concrete', #highly corr w OUTER_WALLS_CAT_Brick_Walls
    'HEATING_TYPE_CAT_Electric_Heating',
    'YEAR',
    'WETROOMS', #highly corr w RESIDENTIAL_AREA
    'HEATING_TYPE_CAT_Stove_Fireplace'
]
top_uncorr = top10_feat + ['OUTER_WALLS_CAT_Timber_Framing', 
                                'HEATING_TYPE_CAT_Central_Heating_Own_System', 
                                'ROOF_TYPE_CAT_Fiber_Cement_Asbestos',
                                'HEATING_TYPE_CAT_Electric_Heating', 
                                'YEAR', 
                                'HEATING_TYPE_CAT_Stove_Fireplace']
top_unique = ['CONSTRUCTION_YEAR', 
              'HOUDEN10KM', 
              'DEDUCTIBLE', 
              'RESIDENTIAL_AREA', 
              'ROOF_TYPE_CAT_Thatched_Roof', 
              'BUILDINGS', 
              'WATER_SUPPLY_TYPE_CAT_Private_Water_Supply', 
              'BASEMENT_AREA', 
              'OUTER_WALLS_CAT_Brick_Walls', 
              'HEATING_TYPE_CAT_Central_Heating_Own_System',
              'YEAR',
              'WETROOMS'] #no two house type variables in same category

data['log_CLAIM_SIZE_INDEX'] = np.log1p(data["CLAIM_SIZE_INDEX"])

In [ ]:
extra_cols = ["log_CLAIM_SIZE_INDEX", "CLAIM_SIZE_INDEX"]

data10 = data[top10_feat + extra_cols]
data20 = data[top20_feat + extra_cols]
data_uncorr = data[top_uncorr + extra_cols]
data_unique = data[top_unique + extra_cols]

print(data20.head(3))

In [25]:
firedata = data

groups = {
    "top10": data[top10_feat],
    "top20": data[top20_feat],
    "uncorr": data[top_uncorr],
    "unique": data[top_unique]
}

TARGETS = {
    "raw": firedata["CLAIM_SIZE_INDEX"],
    "log": np.log1p(firedata["CLAIM_SIZE_INDEX"])
}

# Initial HP grid search

## Tree-based models

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from tqdm.auto import tqdm

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split

def relative_mad(y_true, y_pred, limit):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    eps = 1e-12

    rel_err = np.abs((y_pred - y_true) / (y_true + eps))

    weights = np.where(y_true > limit, 3.292047, 1.0)

    return np.mean(rel_err * weights)



LGBM_GRID = [
    {"num_leaves": 31, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.05},
    {"num_leaves": 63, "learning_rate": 0.1},
]

XGB_GRID = [
    {"max_depth": 4, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.05},
    {"max_depth": 6, "learning_rate": 0.1},
]

CAT_GRID = [
    {"depth": 6, "learning_rate": 0.05},
    {"depth": 8, "learning_rate": 0.05},
    {"depth": 6, "learning_rate": 0.1},
]


n_seeds = 5



def build_lgbm(seed, params):

    return lgb.LGBMRegressor(
        objective="regression",
        n_estimators=1500,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=seed,
        verbose=-1,
        **params
    )


def build_xgb(seed, params):

    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1500,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mae",
        random_state=seed,
        n_jobs=-1,
        **params
    )


def build_cat(seed, params):

    return CatBoostRegressor(
        loss_function="MAE",
        iterations=1500,
        subsample=0.8,
        random_seed=seed,
        verbose=0,
        **params
    )


MODELS = {
    "LightGBM": (build_lgbm, LGBM_GRID),
    "XGBoost": (build_xgb, XGB_GRID),
    "CatBoost": (build_cat, CAT_GRID)
}


results = {}


for model_name, (builder, grid) in MODELS.items():

    print(f"\n\n==============================")
    print(f"MODEL: {model_name}")
    print(f"==============================")

    results[model_name] = {}

    for target_name, y_full in TARGETS.items():

        print(f"\n--- TARGET: {target_name} ---")

        results[model_name][target_name] = []

        for config_id, params in enumerate(grid):

            print(f"\n>>> Config {config_id+1}/{len(grid)}: {params}")

            config_results = {}

            for group_name, X_full in groups.items():

                print(f"  -> Group: {group_name}")

                losses = []

                for seed in range(n_seeds):

                    print(f"     seed {seed+1}/{n_seeds}", end="\r")

                    X_train, X_valid, y_train, y_valid = train_test_split(
                        X_full,
                        y_full,
                        test_size=0.25,
                        random_state=seed
                    )

                    model = builder(seed, params)

                    model.fit(X_train, y_train)

                    pred = model.predict(X_valid)


                    if target_name == "log":
                          pred_eval = np.expm1(pred)
                          y_eval = np.expm1(y_valid)
                          
                    else:
                          pred_eval = pred
                          y_eval = y_valid
                    
                    loss = relative_mad(y_eval, pred_eval, limit)
                    losses.append(loss)

                print("     done                     ")

                config_results[group_name] = np.mean(losses)

            # rank score (lower is better)
            rank_score = sum(
                sorted(config_results.values()).index(v)
                for v in config_results.values()
            )

            results[model_name][target_name].append({
                "params": params,
                "scores": config_results,
                "rank_score": rank_score
            })


rows = []

for model_name in results:

    for target_name in results[model_name]:

        for cfg in results[model_name][target_name]:

            params = cfg["params"]
            scores = cfg["scores"]

            for feature_set, score in scores.items():

                rows.append({
                    "model": model_name,
                    "target": target_name,
                    "feature_set": feature_set,
                    "score": score,
                    **params
                })


df_results = pd.DataFrame(rows)

df_sorted = df_results.sort_values("score", ascending=True)

## NN (MLPRegressor) initial HP grid search

In [ ]:
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


firedata = data

groups = {
    "top10": data[top10_feat],
    "top20": data[top20_feat],
    "uncorr": data[top_uncorr],
    "unique": data[top_unique]
}

TARGETS = {
    "raw": firedata["CLAIM_SIZE_INDEX"],
    "log": np.log1p(firedata["CLAIM_SIZE_INDEX"])}


def relative_mad(y_true, y_pred, limit):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    eps = 1e-12

    rel_err = np.abs((y_pred - y_true) / (y_true + eps))

    weights = np.where(y_true > limit, 3.292047, 1.0)

    return np.mean(rel_err * weights)


MLP_GRID = [

    {
        "hidden_layer_sizes": (64, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },

    {
        "hidden_layer_sizes": (128, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },

    {
        "hidden_layer_sizes": (128, 128),
        "alpha": 1e-3,
        "learning_rate_init": 5e-4
    }
]

n_seeds = 10

def build_mlp(seed, params):

    return Pipeline([

        ("scaler", StandardScaler()),

        ("model", MLPRegressor(
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=seed,
            **params
        ))
    ])

results = []

for target_name, y_full in TARGETS.items():

    print(f"\n==============================")
    print(f"TARGET: {target_name}")
    print(f"==============================")

    for config_id, params in enumerate(MLP_GRID):

        print(f"\n>>> Config {config_id+1}/{len(MLP_GRID)}")
        print(params)

        for group_name, X_full in groups.items():

            print(f"\n  -> Feature set: {group_name}")

            losses = []

            for seed in range(n_seeds):

                print(f"     seed {seed+1}/{n_seeds}", end="\r")

                X_train, X_valid, y_train, y_valid = train_test_split(
                    X_full,
                    y_full,
                    test_size=0.25,
                    random_state=seed
                )

                model = build_mlp(seed, params)

                model.fit(X_train, y_train)

                pred = model.predict(X_valid)

                if target_name == "log":

                    pred_eval = np.expm1(pred)
                    y_eval = np.expm1(y_valid)

                else:

                    pred_eval = pred
                    y_eval = y_valid

                loss = relative_mad(y_eval, pred_eval, limit)

                losses.append(loss)

            mean_loss = np.mean(losses)

            results.append({
                "model": "MLP",
                "target": target_name,
                "feature_set": group_name,
                "score": mean_loss,
                **params
            })

            print(f"     RelMAD = {mean_loss:.6f}")


df_mlp = pd.DataFrame(results)

df_mlp_sorted = (
    df_mlp
    .sort_values("score")
    .reset_index(drop=True)
)

In [ ]:
df_lgbm = df_sorted[df_sorted["model"] == "LightGBM"].copy() 
df_lgbm = df_lgbm.drop(columns=["max_depth", "depth"], errors="ignore") 
print(df_lgbm.head(3))

df_xgb = df_sorted[df_sorted["model"] == "XGBoost"].copy() 
df_xgb = df_xgb.drop(columns=["num_leaves", "depth"], errors="ignore") 
print(df_xgb.head(3))

df_cat = df_sorted[df_sorted["model"] == "CatBoost"].copy() 
df_cat = df_cat.drop(columns=["max_depth", "num_leaves"], errors="ignore") 
print(df_cat.head(3))

print(df_mlp_sorted.head(3))

print(data["log_CLAIM_SIZE_INDEX"].min())
print(data["log_CLAIM_SIZE_INDEX"].max())

# Final HP optimization

Given the best feature set, which hyperparameter optimization strategy (Grid, Random, or Optuna) finds the best CatBoost hyperparameters, and how long does each take?

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from scipy.stats import loguniform
import optuna

winner = df_mlp_sorted.sort_values("score").iloc[0]

feature_map = {
    "top10": top10_feat,
    "top20": top20_feat,
    "uncorr": top_uncorr,
    "unique": top_unique
}

best_feature_set = winner["feature_set"]

X_full = data[feature_map[best_feature_set]].astype(np.float32)

y_full = data["CLAIM_SIZE_INDEX"].values


n_seeds = 10



def relative_mad(y_true, y_pred, limit):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    eps = 1e-12

    rel_err = np.abs((y_pred - y_true) / (y_true + eps))

    weights = np.where(y_true > limit, 3.292047, 1.0)

    return np.mean(rel_err * weights)


limit = np.median(y_full)

def build_mlp(seed, params):

    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPRegressor(
            max_iter=800,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=seed,
            **params
        ))
    ])


def eval_mlp(params):

    losses = []

    for seed in range(n_seeds):

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_full,
            y_full,
            test_size=0.25,
            random_state=seed
        )

        model = build_mlp(seed, params)

        model.fit(X_train, y_train)

        pred = model.predict(X_valid)

        loss = relative_mad(
            y_valid,
            pred,
            limit
        )

        losses.append(loss)

    return np.mean(losses)

MLP_GRID = [
    {
        "hidden_layer_sizes": (64, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 64),
        "alpha": 1e-4,
        "learning_rate_init": 1e-3
    },
    {
        "hidden_layer_sizes": (128, 128),
        "alpha": 1e-3,
        "learning_rate_init": 5e-4
    }
]

start = time.time()

grid_results = []

for params in MLP_GRID:

    score = eval_mlp(params)

    grid_results.append({
        **params,
        "score": score
    })

grid_time = time.time() - start

df_grid = pd.DataFrame(grid_results).sort_values("score")

random_space = {
    "hidden_layer_sizes": [(64, 64), (128, 64), (128, 128)],
    "alpha": loguniform(1e-5, 1e-3),
    "learning_rate_init": loguniform(1e-4, 5e-3)
}

random_params = list(
    ParameterSampler(random_space, n_iter=50, random_state=42)
)

start = time.time()

random_results = []

for params in random_params:

    score = eval_mlp(params)

    random_results.append({
        **params,
        "score": score
    })

random_time = time.time() - start

df_random = pd.DataFrame(random_results).sort_values("score")

def objective(trial):

    params = {
        "hidden_layer_sizes": (
            trial.suggest_categorical("h1", [64, 128]),
            trial.suggest_categorical("h2", [64, 128])
        ),
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-3, log=True),
        "learning_rate_init": trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    }

    return eval_mlp(params)


start = time.time()

study = optuna.create_study(direction="minimize")

study.optimize(
    objective,
    n_trials=50
)

optuna_time = time.time() - start

comparison = pd.DataFrame([
    {
        "method": "Grid",
        "best_score": df_grid["score"].min(),
        "runtime_sec": grid_time
    },
    {
        "method": "Random",
        "best_score": df_random["score"].min(),
        "runtime_sec": random_time
    },
    {
        "method": "Optuna",
        "best_score": study.best_value,
        "runtime_sec": optuna_time
    }
]).sort_values("best_score")


print("\n===== FINAL COMPARISON =====")
print(comparison)

print("\n===== BEST GRID PARAMS =====")
print(df_grid.iloc[0])

print("\n===== BEST RANDOM PARAMS =====")
print(df_random.iloc[0])

print("\n===== BEST OPTUNA PARAMS =====")
print(study.best_params)
print("score:", study.best_value)

## Validation of model with best performing HPs

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

winner = df_mlp_sorted.iloc[0]

feature_map = {
    "top10": top10_feat,
    "top20": top20_feat,
    "uncorr": top_uncorr,
    "unique": top_unique
}

best_feature_set = winner["feature_set"]

X_full = data[
    feature_map[best_feature_set]
].astype(np.float32)

# Use raw target if that was the winning setup
y_full = data["CLAIM_SIZE_INDEX"].values

best_params = {
    "hidden_layer_sizes": (64, 64),
    "alpha": 0.0008898416750158575,
    "learning_rate_init": 0.0006263298840589133
}


n_seeds = 10
seed_scores = []


print("\n==============================")
print("FINAL ROBUST VALIDATION (MLP)")
print("==============================")

print("\nBest feature set:")
print(best_feature_set)

print("\nBest parameters:")
print(best_params)


def rmse(y_true, y_pred):
    return np.sqrt(
        np.mean(
            (y_pred - y_true) ** 2
        )
    )


def build_mlp(seed):

    return Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            MLPRegressor(
                max_iter=800,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=20,
                random_state=seed,
                **best_params
            )
        )
    ])


for seed in range(n_seeds):

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_full,
        y_full,
        test_size=0.25,
        random_state=seed
    )

    model = build_mlp(seed)
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    score = rmse(y_valid, pred)

    seed_scores.append(score)

    print(f"Seed {seed:2d}: "f"RMSE = {score:.6f}")




seed_scores = np.array(seed_scores)

summary = pd.DataFrame({
    "seed": np.arange(n_seeds),
    "score": seed_scores
})

print("\n===== SEED RESULTS =====")
print(summary)

print("\n===== FINAL ESTIMATE =====")
print(f"Mean RMSE : {seed_scores.mean():.6f}")
print(f"Std RMSE  : {seed_scores.std(ddof=1):.6f}")
print(f"Min RMSE  : {seed_scores.min():.6f}")
print(f"Max RMSE  : {seed_scores.max():.6f}")



final_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        MLPRegressor(
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=20,
            random_state=42,
            **best_params
        )
    )
])

final_model.fit(
    X_full,
    y_full
)

print("\nFinal MLP model fitted on full dataset.")